# Coffee & Health — Análise Exploratória de Dados

**UC:** Análise de Dados e Big Data · Projeto A3 · 2026/2

| Nome | RA |
|---|---|
| | |
| | |

## 1. Introdução

Contexto, apresentação do dataset e objetivos do trabalho.

> Os dados são **sintéticos**: os resultados mostram associações, não causa e efeito nem evidência clínica.

## 2. Bibliotecas e configuração

| Biblioteca | Uso no projeto |
|---|---|
| `pandas` | Leitura do CSV e manipulação dos dados em DataFrames |
| `matplotlib` | Base dos gráficos |
| `seaborn` | Gráficos estatísticos sobre o matplotlib |
| `scikit-learn` | Separação treino/teste, Árvore de Decisão e métricas de classificação (Q8) |
| `python-dotenv` | Leitura dos valores fixos guardados no arquivo `.env` |
| `os` e `pathlib` | Variáveis de ambiente e caminhos de arquivo |

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

### Configuração

Os valores fixos do projeto ficam no arquivo `.env` (veja `.env.example`). O valor ao lado de cada variável só é usado quando o `.env` não existe.

In [ ]:
# Raiz do repositório: o notebook pode ser executado a partir de notebooks/ ou da raiz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(RAIZ / ".env")

DATA_PATH = RAIZ / os.getenv("DATA_PATH", "data/synthetic_coffee_health_10000.csv")
RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))
TEST_SIZE = float(os.getenv("TEST_SIZE", "0.2"))
TOP_N_PAISES = int(os.getenv("TOP_N_PAISES", "5"))
FIG_LARGURA = float(os.getenv("FIG_LARGURA", "10"))
FIG_ALTURA = float(os.getenv("FIG_ALTURA", "5"))
CASAS_DECIMAIS = int(os.getenv("CASAS_DECIMAIS", "2"))
CASAS_CORRELACAO = int(os.getenv("CASAS_CORRELACAO", "4"))
MARGEM_TOPO_GRAFICO = float(os.getenv("MARGEM_TOPO_GRAFICO", "0.18"))
PADDING_ROTULO = int(os.getenv("PADDING_ROTULO", "4"))
BINS_HISTOGRAMA = int(os.getenv("BINS_HISTOGRAMA", "20"))
ALPHA_DISPERSAO = float(os.getenv("ALPHA_DISPERSAO", "0.25"))
TAMANHO_PONTO = int(os.getenv("TAMANHO_PONTO", "15"))
MAX_PROFUNDIDADE_ARVORE = int(os.getenv("MAX_PROFUNDIDADE_ARVORE", "3"))
FAIXAS_ETARIAS = [int(limite) for limite in os.getenv("FAIXAS_ETARIAS", "18,30,45,81").split(",")]
K_ANONIMATO = int(os.getenv("K_ANONIMATO", "5"))

# Aparência padrão dos gráficos e das tabelas
sns.set_theme(style="whitegrid", rc={"figure.figsize": (FIG_LARGURA, FIG_ALTURA)})
pd.set_option("display.max_columns", None)

## 3. Carga dos dados

O CSV original nunca é alterado: `df_original` guarda os dados como foram lidos e `df` é a cópia usada nas transformações.

> Na leitura, o pandas converte o texto `None` da coluna `Health_Issues` em valor ausente (`NaN`). Esses registros são tratados na Q2.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"CSV não encontrado em {DATA_PATH}. Confira DATA_PATH no arquivo .env.")

df_original = pd.read_csv(DATA_PATH)
df = df_original.copy()

print(f"Arquivo carregado: {DATA_PATH.name}")
print(f"{df.shape[0]} registros e {df.shape[1]} colunas")

## 4. Q1 — Conhecendo o dataset

Quantos registros e colunas existem, quais são os tipos de dados, as cinco primeiras linhas e o resumo estatístico das colunas numéricas.

### Registros e colunas

In [ ]:
n_registros, n_colunas = df.shape

print(f"Registros: {n_registros}")
print(f"Colunas: {n_colunas}")

### Tipos de dados

`int64` são números inteiros, `float64` são números decimais e `str` (ou `object`, em versões mais antigas do pandas) são textos.

In [ ]:
df.dtypes.to_frame("Tipo no pandas")

In [ ]:
df.dtypes.astype(str).value_counts().to_frame("Quantidade de colunas")

### Cinco primeiras linhas

In [ ]:
df.head()

### Resumo estatístico das colunas numéricas

O `describe()` considera todas as colunas com tipo numérico. Três delas não são medidas:

- `ID` é só um identificador, então média e desvio padrão não têm significado;
- `Smoking` e `Alcohol_Consumption` são indicadores `0`/`1`, e a média equivale à proporção de registros com valor `1`.

In [ ]:
df.describe()

### Classificação das variáveis

O tipo no pandas indica como o dado está armazenado, não o que ele representa. A tabela abaixo classifica cada coluna pelo tipo de dado estatístico:

- **Numérica contínua:** pode assumir valores decimais dentro de um intervalo;
- **Numérica discreta:** valores inteiros, resultado de contagem ou registrados sem casas decimais;
- **Categórica nominal:** categorias sem ordem;
- **Categórica ordinal:** categorias com ordem (`Poor` < `Fair` < `Good` < `Excellent`; `Low` < `Medium` < `High`; `Mild` < `Moderate` < `Severe`);
- **Binária:** só dois valores, `0` e `1`;
- **Identificador:** distingue os registros e não entra nas análises.

In [ ]:
tipos_de_dado = {
    "ID": "Identificador",
    "Age": "Numérica discreta",
    "Gender": "Categórica nominal",
    "Country": "Categórica nominal",
    "Coffee_Intake": "Numérica contínua",
    "Caffeine_mg": "Numérica contínua",
    "Sleep_Hours": "Numérica contínua",
    "Sleep_Quality": "Categórica ordinal",
    "BMI": "Numérica contínua",
    "Heart_Rate": "Numérica discreta",
    "Stress_Level": "Categórica ordinal",
    "Physical_Activity_Hours": "Numérica contínua",
    "Health_Issues": "Categórica ordinal",
    "Occupation": "Categórica nominal",
    "Smoking": "Binária",
    "Alcohol_Consumption": "Binária",
}

classificacao = pd.DataFrame({
    "Tipo no pandas": df.dtypes.astype(str),
    "Valores distintos": df.nunique(),
    "Classificação": pd.Series(tipos_de_dado),
})
classificacao

In [ ]:
classificacao["Classificação"].value_counts().to_frame("Quantidade de colunas")

### Resposta

- O dataset tem **10.000 registros** e **16 colunas**; cada registro representa uma pessoa.
- No pandas, 5 colunas são inteiras (`int64`), 5 são decimais (`float64`) e 6 são de texto.
- Pela classificação estatística, são 5 numéricas contínuas, 2 numéricas discretas, 3 categóricas nominais, 3 categóricas ordinais, 2 binárias e 1 identificador.
- A idade média é de cerca de 35 anos, o consumo médio de café é de 2,5 e a média de sono é de 6,6 horas.
- A média de `Smoking` (0,20) e de `Alcohol_Consumption` (0,30) mostra que cerca de 20% dos registros são de fumantes e 30% de pessoas que consomem álcool.
- `Health_Issues` já aparece com valores ausentes (`NaN`) nas primeiras linhas; eles são analisados na Q2.

## 5. Q2 — Qualidade dos dados e ETL

O dataset tem valores ausentes ou registros duplicados? Em quais colunas, em que quantidade e como tratá-los sem modificar o arquivo original.

O diagnóstico é feito sobre `df_original`, que guarda os dados exatamente como foram lidos do CSV.

### Valores ausentes

Um valor ausente indica que a informação não foi registrada. No pandas, ele aparece como `NaN`.

- `isnull()` devolve `True` em cada célula vazia e `False` nas preenchidas.
- `.sum()` soma esses valores por coluna (`True` vale 1), resultando na quantidade de ausentes.
- `.mean()` calcula a proporção de ausentes em cada coluna; multiplicada por 100, vira percentual.

> **Atenção:** no CSV, os ausentes de `Health_Issues` estão escritos como o texto `None`. O `read_csv` converte esse texto em `NaN` automaticamente. Ausência **não** significa "sem problemas de saúde"; significa apenas que o dado não foi informado.

In [ ]:
ausentes = pd.DataFrame({
    "Valores ausentes": df_original.isnull().sum(),
    "% do total": df_original.isnull().mean() * 100,
})
ausentes

### Registros duplicados

Três verificações: linhas inteiras repetidas, linhas repetidas quando o `ID` é ignorado (a mesma pessoa cadastrada duas vezes com identificadores diferentes) e identificadores repetidos.

In [ ]:
linhas_duplicadas = df_original.duplicated().sum()
linhas_duplicadas_sem_id = df_original.drop(columns="ID").duplicated().sum()
ids_repetidos = df_original["ID"].duplicated().sum()

print(f"Linhas duplicadas: {linhas_duplicadas}")
print(f"Linhas duplicadas ignorando o ID: {linhas_duplicadas_sem_id}")
print(f"IDs repetidos: {ids_repetidos}")

### Consistência das categorias

Confere se as colunas de texto têm só as categorias previstas no dicionário de dados, sem variações de grafia.

In [ ]:
colunas_categoricas = ["Gender", "Country", "Sleep_Quality", "Stress_Level", "Health_Issues", "Occupation"]

for coluna in colunas_categoricas:
    categorias = sorted(df_original[coluna].dropna().unique())
    print(f"{coluna} ({len(categorias)}): {categorias}")

### Fluxo ETL

```
┌──────────────┐      ┌────────────────────────────┐      ┌──────────────────┐
│   EXTRAÇÃO   │      │       TRANSFORMAÇÃO        │      │      CARGA       │
│              │ ───► │                            │ ───► │                  │
│  CSV → df_   │      │  cópia → duplicados →      │      │  df: DataFrame   │
│  original    │      │  ausentes → ordinais →     │      │  limpo, usado    │
│              │      │  anonimização              │      │  nas análises    │
└──────────────┘      └────────────────────────────┘      └──────────────────┘
```

| Fase | Passo | O que é feito |
|---|---|---|
| Extração | Leitura | `pd.read_csv` lê o arquivo para `df_original` (seção 3). O CSV nunca é regravado. |
| Transformação | 1. Cópia | `df_original.copy()`: todas as mudanças são feitas na cópia. |
| Transformação | 2. Duplicados | `drop_duplicates()` remove linhas repetidas, se existirem. |
| Transformação | 3. Ausentes | Os vazios de `Health_Issues` recebem o rótulo `Não informado`. |
| Transformação | 4. Ordinais | `Sleep_Quality`, `Stress_Level` e `Health_Issues` passam a ter as categorias na ordem correta. |
| Transformação | 5. Anonimização | A coluna `ID` é removida, porque identifica o registro e não é usada nas análises. |
| Carga | DataFrame limpo | O resultado fica em `df`, usado em todas as seções seguintes. |

### Transformação

In [ ]:
ROTULO_NAO_INFORMADO = "Não informado"
ORDEM_SLEEP_QUALITY = ["Poor", "Fair", "Good", "Excellent"]
ORDEM_STRESS_LEVEL = ["Low", "Medium", "High"]
ORDEM_HEALTH_ISSUES = ["Mild", "Moderate", "Severe"]

# 1. Cópia: o original fica preservado em df_original
df = df_original.copy()

# 2. Duplicados
df = df.drop_duplicates()

# 3. Ausentes: vazio não significa "sem problema de saúde"
df["Health_Issues"] = df["Health_Issues"].fillna(ROTULO_NAO_INFORMADO)

# 4. Ordinais: categorias na ordem correta
df["Sleep_Quality"] = pd.Categorical(df["Sleep_Quality"], categories=ORDEM_SLEEP_QUALITY, ordered=True)
df["Stress_Level"] = pd.Categorical(df["Stress_Level"], categories=ORDEM_STRESS_LEVEL, ordered=True)
df["Health_Issues"] = pd.Categorical(df["Health_Issues"], categories=ORDEM_HEALTH_ISSUES + [ROTULO_NAO_INFORMADO])

# 5. Anonimização: remove o identificador
df = df.drop(columns="ID").reset_index(drop=True)

### Carga

Conferência do DataFrame limpo e de que o original continua intacto.

In [ ]:
print(f"Original: {df_original.shape[0]} registros e {df_original.shape[1]} colunas")
print(f"Limpo:    {df.shape[0]} registros e {df.shape[1]} colunas")
print(f"Valores ausentes no DataFrame limpo: {df.isnull().sum().sum()}")
print(f"Valores ausentes no original: {df_original.isnull().sum().sum()}")

In [ ]:
df["Health_Issues"].value_counts().to_frame("Registros")

In [ ]:
df.head()

### Resposta

- **Valores ausentes:** só a coluna `Health_Issues` tem valores ausentes: **5.941 registros (59,41%)**. As outras 15 colunas estão completas.
- **Duplicados:** não há registros duplicados, nem linhas inteiras, nem linhas iguais com `ID` diferente, nem `ID` repetido.
- **Categorias:** as colunas de texto têm só as categorias previstas, sem variações de grafia.

**Como tratar sem modificar o arquivo original:** o CSV é apenas lido, e todas as mudanças são feitas em `df`, uma cópia de `df_original`. Para os vazios de `Health_Issues` há duas opções, usadas conforme a análise:

1. **Rotular como `Não informado`** (feito acima): mantém os 10.000 registros nas análises que não dependem dessa coluna.
2. **Filtrar só os registros preenchidos**: é o que a Q10 pede, com `df[df["Health_Issues"] != ROTULO_NAO_INFORMADO]`.

Não é adequado preencher os vazios com a categoria mais frequente (`Mild`) nem tratá-los como "sem problema de saúde". No CSV esses campos trazem o texto `None`, mas o dicionário de dados orienta a não presumir que ausência significa ausência de problema, e alterar quase 60% da coluna distorceria qualquer comparação.

> **Para as próximas questões:** `Sleep_Quality`, `Stress_Level` e `Health_Issues` agora são colunas categóricas. Ao agrupar por elas, use `groupby(..., observed=True)` para que só apareçam as categorias e combinações que existem nos dados.

## 6. Análise Exploratória (EDA)

### Q3 — Perfil das pessoas

Qual é a distribuição dos registros por gênero (`Gender`) e qual é a média de idade (`Age`) em cada gênero?

#### Distribuição por gênero

In [ ]:
contagem_genero = df["Gender"].value_counts()
percentual_genero = df["Gender"].value_counts(normalize=True) * 100

distribuicao_genero = pd.DataFrame({
    "Registros": contagem_genero,
    "% do total": percentual_genero.round(CASAS_DECIMAIS),
})
distribuicao_genero

#### Idade por gênero

Além da média, a tabela traz a mediana, o desvio padrão e os extremos de idade em cada gênero. Comparar média e mediana ajuda a verificar se valores extremos puxam a média.

In [ ]:
ordem_generos = distribuicao_genero.index

idade_por_genero = (
    df.groupby("Gender")["Age"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={
        "count": "Registros",
        "mean": "Média",
        "median": "Mediana",
        "std": "Desvio padrão",
        "min": "Mínima",
        "max": "Máxima",
    })
    .reindex(ordem_generos)
    .round(CASAS_DECIMAIS)
)
idade_por_genero

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
cores_genero = sns.color_palette(n_colors=len(distribuicao_genero))

# Gráfico 1: quantidade e percentual de registros por gênero
barras_qtd = ax1.bar(distribuicao_genero.index, distribuicao_genero["Registros"], color=cores_genero)
rotulos_qtd = [
    f"{int(linha['Registros'])}\n({linha['% do total']:.{CASAS_DECIMAIS}f}%)"
    for _, linha in distribuicao_genero.iterrows()
]
ax1.bar_label(barras_qtd, labels=rotulos_qtd, padding=PADDING_ROTULO)

# Gráfico 2: idade média por gênero com referência da média geral
barras_idade = ax2.bar(idade_por_genero.index, idade_por_genero["Média"], color=cores_genero)
ax2.bar_label(barras_idade, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)

media_geral_idade = df["Age"].mean()
ax2.axhline(
    media_geral_idade,
    color="gray",
    linestyle="--",
    label=f"Média geral ({media_geral_idade:.{CASAS_DECIMAIS}f} anos)",
)
ax2.legend(loc="upper right")

config_graficos = [
    (ax1, "Distribuição de registros por gênero", "Quantidade de registros", distribuicao_genero["Registros"].max()),
    (ax2, "Idade média por gênero", "Idade média (anos)", idade_por_genero["Média"].max()),
]
for ax, titulo, rotulo_y, valor_max in config_graficos:
    ax.set_title(titulo)
    ax.set_xlabel("Gênero")
    ax.set_ylabel(rotulo_y)
    ax.set_ylim(0, valor_max * (1 + MARGEM_TOPO_GRAFICO))
    ax.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Distribuição por gênero:** entre os 10.000 registros, `Female` concentra a maior parte (**5.001** registros; **50,01%**), seguido de perto por `Male` (**4.773**; **47,73%**), enquanto `Other` representa uma parcela menor (**226**; **2,26%**).
- **Média de idade por gênero:** as médias são muito próximas entre os três grupos — `Female` tem média de **35,03 anos**, `Male` de **34,89 anos** e `Other` de **34,50 anos** (diferença de cerca de 0,5 ano entre o maior e o menor grupo, próxima à média geral de **34,95 anos**).
- **Média vs. mediana:** em todos os gêneros, a mediana (`35` anos em `Female` e `34` anos em `Male` e `Other`) fica muito próxima da média, indicando distribuição etária equilibrada e sem distorção forte por valores extremos (idades de `18` a `80` anos em `Female` e `Male`, e de `18` a `64` em `Other`).

### Q4 — Consumo de café

Qual é o valor médio, mínimo e máximo de `Coffee_Intake`? Quais são os cinco países (`Country`) com maior média de consumo de café no dataset?

#### Estatísticas gerais de consumo

In [ ]:
resumo_cafe = pd.Series({
    "Média": df["Coffee_Intake"].mean(),
    "Mediana": df["Coffee_Intake"].median(),
    "Desvio padrão": df["Coffee_Intake"].std(),
    "Mínimo": df["Coffee_Intake"].min(),
    "Máximo": df["Coffee_Intake"].max(),
}).round(CASAS_DECIMAIS).to_frame("Coffee_Intake")

registros_zero = df["Coffee_Intake"].eq(0).sum()
print(f"Registros com consumo igual a zero: {registros_zero} ({registros_zero / len(df) * 100:.{CASAS_DECIMAIS}f}%)")
resumo_cafe

#### Países com maior média de consumo

In [ ]:
consumo_por_pais = (
    df.groupby("Country")["Coffee_Intake"]
    .agg(["count", "mean", "median", "std"])
    .rename(columns={
        "count": "Registros",
        "mean": "Média",
        "median": "Mediana",
        "std": "Desvio padrão",
    })
    .sort_values("Média", ascending=False)
    .round(CASAS_DECIMAIS)
)

top_paises = consumo_por_pais.head(TOP_N_PAISES)
print(f"Total de países: {len(consumo_por_pais)} | Menor média: {consumo_por_pais.index[-1]} ({consumo_por_pais['Média'].iloc[-1]:.{CASAS_DECIMAIS}f})")
top_paises

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))

# Gráfico 1: distribuição do consumo de café com média e mediana
ax1.hist(df["Coffee_Intake"], bins=BINS_HISTOGRAMA, edgecolor="white")
media_cafe = df["Coffee_Intake"].mean()
mediana_cafe = df["Coffee_Intake"].median()
ax1.axvline(media_cafe, color="crimson", linestyle="--", label=f"Média ({media_cafe:.{CASAS_DECIMAIS}f})")
ax1.axvline(mediana_cafe, color="darkorange", linestyle=":", label=f"Mediana ({mediana_cafe:.{CASAS_DECIMAIS}f})")
ax1.set_title("Distribuição de Coffee_Intake")
ax1.set_xlabel("Consumo de café (Coffee_Intake)")
ax1.set_ylabel("Quantidade de registros")
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)

# Gráfico 2: top países por consumo médio
top_ordenado = top_paises.iloc[::-1]
barras_paises = ax2.barh(top_ordenado.index, top_ordenado["Média"])
ax2.bar_label(barras_paises, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax2.set_title(f"Top {TOP_N_PAISES} países por média de consumo")
ax2.set_xlabel("Média de Coffee_Intake")
ax2.set_ylabel("País")
ax2.set_xlim(0, top_paises["Média"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="y", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Resumo de `Coffee_Intake`:** média de **2,51**, mediana de **2,50**, valor mínimo de **0,0** (`558` registros, ou `5,58%`) e máximo de **8,2**. A proximidade entre média e mediana mostra distribuição equilibrada, sem forte distorção por valores extremos.
- **Cinco países com maior média:** **1º `Spain` (`2,60`)**, **2º `Norway` (`2,59`)**, **3º `UK` (`2,58`)**, **4º `China` (`2,57`)** e **5º `USA` (`2,56`)**.
- **Amplitude entre países:** as médias dos 20 países variam em um intervalo curto, de `2,41` (`Belgium`) a `2,60` (`Spain`), indicando consumo médio semelhante entre os países neste dataset sintético.

### Q5 — Café e cafeína

Existe relação entre `Coffee_Intake` e `Caffeine_mg`? Calcule a correlação e represente as variáveis em um gráfico de dispersão. Explique o resultado.

#### Correlação de Pearson

In [ ]:
matriz_corr = df[["Coffee_Intake", "Caffeine_mg"]].corr().round(CASAS_CORRELACAO)
correlacao_cafe_cafeina = df["Coffee_Intake"].corr(df["Caffeine_mg"])

consumo_positivo = df[df["Coffee_Intake"] > 0]
mg_por_unidade = (consumo_positivo["Caffeine_mg"] / consumo_positivo["Coffee_Intake"]).mean()

print(f"Correlação de Pearson: {correlacao_cafe_cafeina:.{CASAS_CORRELACAO}f}")
print(f"Razão média (entre registros com consumo > 0): {mg_por_unidade:.{CASAS_DECIMAIS}f} mg por unidade de Coffee_Intake")
matriz_corr

#### Gráfico de dispersão

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_LARGURA, FIG_ALTURA))

sns.regplot(
    data=df,
    x="Coffee_Intake",
    y="Caffeine_mg",
    scatter_kws={"alpha": ALPHA_DISPERSAO, "s": TAMANHO_PONTO},
    line_kws={"color": "crimson"},
    ax=ax,
)
ax.set_title(f"Relação entre consumo de café e cafeína (r = {correlacao_cafe_cafeina:.{CASAS_CORRELACAO}f})")
ax.set_xlabel("Consumo de café (Coffee_Intake)")
ax.set_ylabel("Cafeína (mg)")

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Correlação encontrada:** a correlação de Pearson entre `Coffee_Intake` e `Caffeine_mg` é **r = 0,9998**, uma relação linear positiva praticamente perfeita.
- **Explicação do resultado:** os pontos do gráfico de dispersão acompanham uma linha reta crescente com dispersão mínima em torno da tendência. Cada unidade de `Coffee_Intake` corresponde, em média, a cerca de **95 mg de cafeína** (valor próximo ao de uma xícara padrão), indicando que `Caffeine_mg` foi gerada de forma diretamente proporcional ao consumo de café no dataset sintético e traz praticamente a mesma informação.

### Q6 — Consumo de café e sono

Qual é a média de `Coffee_Intake` para cada categoria de `Sleep_Quality`? Os registros classificados com sono `Poor` apresentam consumo médio maior ou menor do que aqueles com sono `Excellent`?

#### Consumo por qualidade do sono

In [ ]:
cafe_por_sono = (
    df.groupby("Sleep_Quality", observed=True)["Coffee_Intake"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={
        "count": "Registros",
        "mean": "Média",
        "median": "Mediana",
        "std": "Desvio padrão",
        "min": "Mínimo",
        "max": "Máximo",
    })
    .round(CASAS_DECIMAIS)
)

dif_poor_excellent = cafe_por_sono.loc["Poor", "Média"] - cafe_por_sono.loc["Excellent", "Média"]
print(f"Diferença média (Poor - Excellent): {dif_poor_excellent:.{CASAS_DECIMAIS}f}")
cafe_por_sono

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
cores_sono = sns.color_palette(n_colors=len(cafe_por_sono))

# Gráfico 1: média de Coffee_Intake por Sleep_Quality
barras_sono = ax1.bar(cafe_por_sono.index.astype(str), cafe_por_sono["Média"], color=cores_sono)
ax1.bar_label(barras_sono, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.axhline(
    df["Coffee_Intake"].mean(),
    color="gray",
    linestyle="--",
    label=f"Média geral ({df['Coffee_Intake'].mean():.{CASAS_DECIMAIS}f})",
)
ax1.set_title("Consumo médio de café por qualidade do sono")
ax1.set_xlabel("Qualidade do sono")
ax1.set_ylabel("Média de Coffee_Intake")
ax1.set_ylim(0, cafe_por_sono["Média"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)

# Gráfico 2: distribuição (boxplot) para comparar medianas e dispersão
sns.boxplot(data=df, x="Sleep_Quality", y="Coffee_Intake", hue="Sleep_Quality", dodge=False, legend=False, ax=ax2)
ax2.set_title("Distribuição do consumo por qualidade do sono")
ax2.set_xlabel("Qualidade do sono")
ax2.set_ylabel("Coffee_Intake")
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Média de `Coffee_Intake` por categoria:**
  - `Poor` (`961` registros): média **2,98** (mediana **3,00**)
  - `Fair` (`2.050` registros): média **2,76** (mediana **2,80**)
  - `Good` (`5.637` registros): média **2,45** (mediana **2,40**)
  - `Excellent` (`1.352` registros): média **2,05** (mediana **2,00**)
- **Comparação `Poor` vs. `Excellent`:** os registros com sono **`Poor`** apresentam consumo médio **maior** do que aqueles com sono **`Excellent`** (`2,98` contra `2,05`, uma diferença de **0,93** unidade).
- **Padrão e outliers:** a média cai de forma contínua conforme a qualidade do sono melhora, e a mediana acompanha a média de perto em todas as faixas, mostrando que a diferença reflete o comportamento geral dos grupos e não apenas valores extremos.

### Q7 — Sono e estresse

Qual é a média de horas de sono (`Sleep_Hours`) em cada nível de estresse (`Stress_Level`)? Apresente um gráfico comparativo e descreva o padrão encontrado.

#### Horas de sono por nível de estresse

In [ ]:
sono_por_estresse = (
    df.groupby("Stress_Level", observed=True)["Sleep_Hours"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={
        "count": "Registros",
        "mean": "Média",
        "median": "Mediana",
        "std": "Desvio padrão",
        "min": "Mínimo",
        "max": "Máximo",
    })
    .round(CASAS_DECIMAIS)
)
sono_por_estresse

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
cores_estresse = sns.color_palette(n_colors=len(sono_por_estresse))

# Gráfico 1: média de horas de sono por nível de estresse
barras_estresse = ax1.bar(sono_por_estresse.index.astype(str), sono_por_estresse["Média"], color=cores_estresse)
ax1.bar_label(barras_estresse, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.axhline(
    df["Sleep_Hours"].mean(),
    color="gray",
    linestyle="--",
    label=f"Média geral ({df['Sleep_Hours'].mean():.{CASAS_DECIMAIS}f} h)",
)
ax1.set_title("Média de horas de sono por nível de estresse")
ax1.set_xlabel("Nível de estresse")
ax1.set_ylabel("Média de Sleep_Hours (h)")
ax1.set_ylim(0, sono_por_estresse["Média"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)

# Gráfico 2: distribuição das horas de sono por nível de estresse
sns.boxplot(data=df, x="Stress_Level", y="Sleep_Hours", hue="Stress_Level", dodge=False, legend=False, ax=ax2)
ax2.set_title("Distribuição de horas de sono por estresse")
ax2.set_xlabel("Nível de estresse")
ax2.set_ylabel("Sleep_Hours (h)")
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Média de `Sleep_Hours` por `Stress_Level`:**
  - `Low` (`6.989` registros): média **7,25 h** (mediana **7,10 h**; de `6,0` a `10,0 h`)
  - `Medium` (`2.050` registros): média **5,58 h** (mediana **5,60 h**; de `5,0` a `6,0 h`)
  - `High` (`961` registros): média **4,45 h** (mediana **4,60 h**; de `3,0` a `5,0 h`)
- **Padrão encontrado:** há uma associação inversa clara e acentuada — conforme o nível de estresse aumenta de `Low` para `High`, a média de sono cai **2,80 horas**. Como mostram os boxplots e os limites mínimo e máximo, os três níveis ocupam faixas praticamente separadas de horas de sono no dataset sintético, com média e mediana muito próximas em todos os grupos.

### Q8 — Classificação com Árvore de Decisão

Modelo simples com `DecisionTreeClassifier` para prever `Sleep_Quality` a partir de `Coffee_Intake`, `Caffeine_mg`, `Age`, `BMI`, `Heart_Rate` e `Physical_Activity_Hours`, sem utilizar `ID`.

#### Separação em treino e teste

In [ ]:
ATRIBUTOS_Q8 = [
    "Coffee_Intake",
    "Caffeine_mg",
    "Age",
    "BMI",
    "Heart_Rate",
    "Physical_Activity_Hours",
]
ALVO_Q8 = "Sleep_Quality"

# Confirma que o identificador não entra nos preditores
assert "ID" not in ATRIBUTOS_Q8 and "ID" not in df.columns

X = df[ATRIBUTOS_Q8]
y = df[ALVO_Q8]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

classe_majoritaria = y_train.mode()[0]
acuracia_baseline = (y_test == classe_majoritaria).mean()

print(f"Treino: {len(X_train)} registros | Teste: {len(X_test)} registros")
print(f"Baseline (sempre prever '{classe_majoritaria}'): {acuracia_baseline * 100:.{CASAS_DECIMAIS}f}%")

#### Treinamento e comparação de desempenho

Além da árvore padrão (`max_depth=None`), testamos uma árvore com profundidade limitada (`MAX_PROFUNDIDADE_ARVORE`) para observar o efeito do ajuste excessivo (*overfitting*) aos dados de treino.

In [ ]:
modelo_livre = DecisionTreeClassifier(random_state=RANDOM_STATE)
modelo_livre.fit(X_train, y_train)

modelo_podado = DecisionTreeClassifier(max_depth=MAX_PROFUNDIDADE_ARVORE, random_state=RANDOM_STATE)
modelo_podado.fit(X_train, y_train)

comparacao_modelos = pd.DataFrame([
    {
        "Modelo": f"Baseline (classe majoritária: {classe_majoritaria})",
        "Acurácia treino (%)": (y_train == classe_majoritaria).mean() * 100,
        "Acurácia teste (%)": acuracia_baseline * 100,
    },
    {
        "Modelo": "Árvore de Decisão (max_depth=None)",
        "Acurácia treino (%)": accuracy_score(y_train, modelo_livre.predict(X_train)) * 100,
        "Acurácia teste (%)": accuracy_score(y_test, modelo_livre.predict(X_test)) * 100,
    },
    {
        "Modelo": f"Árvore de Decisão (max_depth={MAX_PROFUNDIDADE_ARVORE})",
        "Acurácia treino (%)": accuracy_score(y_train, modelo_podado.predict(X_train)) * 100,
        "Acurácia teste (%)": accuracy_score(y_test, modelo_podado.predict(X_test)) * 100,
    },
]).round(CASAS_DECIMAIS)

comparacao_modelos

In [ ]:
pred_podada = modelo_podado.predict(X_test)
print(classification_report(y_test, pred_podada, labels=ORDEM_SLEEP_QUALITY, zero_division=0))

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))

# Gráfico 1: matriz de confusão do modelo podado no conjunto de teste
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_podada,
    labels=ORDEM_SLEEP_QUALITY,
    ax=ax1,
    colorbar=False,
    cmap="Blues",
)
ax1.set_title(f"Matriz de confusão (max_depth={MAX_PROFUNDIDADE_ARVORE})")
ax1.set_xlabel("Previsto")
ax1.set_ylabel("Real")
ax1.grid(False)

# Gráfico 2: importância dos atributos no modelo podado
importancias = pd.Series(modelo_podado.feature_importances_, index=ATRIBUTOS_Q8).sort_values() * 100
barras_imp = ax2.barh(importancias.index, importancias.values)
ax2.bar_label(barras_imp, fmt=f"%.{CASAS_DECIMAIS}f%%", padding=PADDING_ROTULO)
ax2.set_title(f"Importância dos atributos (max_depth={MAX_PROFUNDIDADE_ARVORE})")
ax2.set_xlabel("Importância (%)")
ax2.set_xlim(0, importancias.max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="y", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Acurácia obtida:** com divisão estratificada (`80%` treino / `20%` teste, `RANDOM_STATE=42`), a Árvore de Decisão padrão (`max_depth=None`) atingiu **100,00%** no treino e **38,90%** no teste, evidenciando *overfitting*. Ao limitar a profundidade (`max_depth=3`), a acurácia no teste sobe para **56,40%** (`56,51%` no treino).
- **Interpretação:** a acurácia de **56,40%** apenas empata com a *baseline* majoritária (prever sempre `Good`, que representa `56,37%` dos dados). Como mostra a matriz de confusão, mesmo usando `Caffeine_mg` (`88,38%` da importância na árvore podada), o modelo quase sempre prevê `Good`.
- **Por que isso ocorre:** nos dados sintéticos, `Sleep_Quality` está ligada principalmente a `Sleep_Hours` (que não entra entre os seis preditores da Q8), enquanto `Coffee_Intake`, `Caffeine_mg`, `Age`, `BMI`, `Heart_Rate` e `Physical_Activity_Hours` se sobrepõem fortemente entre as quatro classes.

### Q9 — Análise conjunta de consumo, sono e estresse

Agrupamento por `Stress_Level` e `Sleep_Quality` com quantidade de pessoas, média de `Coffee_Intake` e média de `Sleep_Hours`.

#### Agrupamento por estresse e qualidade do sono

In [ ]:
analise_conjunta = (
    df.groupby(["Stress_Level", "Sleep_Quality"], observed=True)
    .agg(
        Pessoas=("Coffee_Intake", "count"),
        **{
            "Café (média)": ("Coffee_Intake", "mean"),
            "Café (mediana)": ("Coffee_Intake", "median"),
            "Sono h (média)": ("Sleep_Hours", "mean"),
            "Sono h (mediana)": ("Sleep_Hours", "median"),
        },
    )
    .assign(**{"% do total": lambda t: t["Pessoas"] / len(df) * 100})
    .sort_values("Café (média)", ascending=False)
    .round(CASAS_DECIMAIS)
)

total_combinacoes = len(ORDEM_STRESS_LEVEL) * len(ORDEM_SLEEP_QUALITY)
print(f"Combinações existentes nos dados: {len(analise_conjunta)} de {total_combinacoes} possíveis")
analise_conjunta

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))

# Gráfico 1: mapa de calor com média de café e n em cada combinação possível
tabela_calor = df.pivot_table(index="Stress_Level", columns="Sleep_Quality", values="Coffee_Intake", aggfunc="mean", observed=False)
contagem_calor = df.pivot_table(index="Stress_Level", columns="Sleep_Quality", values="Coffee_Intake", aggfunc="count", observed=False)
anotacoes = tabela_calor.copy().astype(object)
for linha in tabela_calor.index:
    for coluna in tabela_calor.columns:
        valor = tabela_calor.loc[linha, coluna]
        qtd = int(contagem_calor.loc[linha, coluna])
        anotacoes.loc[linha, coluna] = f"{valor:.{CASAS_DECIMAIS}f}\n(n={qtd})" if pd.notna(valor) else "—"

sns.heatmap(tabela_calor, annot=anotacoes, fmt="", cmap="YlOrBr", cbar=False, linewidths=1, linecolor="white", ax=ax1)
ax1.set_title("Média de café e contagem (Estresse × Sono)")
ax1.set_xlabel("Qualidade do sono")
ax1.set_ylabel("Nível de estresse")

# Gráfico 2: comparação direta dos 4 grupos existentes
rotulos_grupos = [f"{est}\n{sono}" for est, sono in analise_conjunta.index]
cores_q9 = sns.color_palette(n_colors=len(analise_conjunta))
barras_q9 = ax2.bar(rotulos_grupos, analise_conjunta["Café (média)"], color=cores_q9)
rotulos_q9 = [
    f"{linha['Café (média)']:.{CASAS_DECIMAIS}f}\n(n={int(linha['Pessoas'])})"
    for _, linha in analise_conjunta.iterrows()
]
ax2.bar_label(barras_q9, labels=rotulos_q9, padding=PADDING_ROTULO)
ax2.set_title("Consumo médio de café nos grupos existentes")
ax2.set_xlabel("Nível de estresse / Qualidade do sono")
ax2.set_ylabel("Média de Coffee_Intake")
ax2.set_ylim(0, analise_conjunta["Café (média)"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Grupos existentes:** das 12 combinações possíveis entre `Stress_Level` e `Sleep_Quality`, apenas **4** aparecem nos dados:
  1. `High` × `Poor`: **961** pessoas (`9,61%`) | café médio **2,98** | sono médio **4,45 h**
  2. `Medium` × `Fair`: **2.050** pessoas (`20,50%`) | café médio **2,76** | sono médio **5,58 h**
  3. `Low` × `Good`: **5.637** pessoas (`56,37%`) | café médio **2,45** | sono médio **6,92 h**
  4. `Low` × `Excellent`: **1.352** pessoas (`13,52%`) | café médio **2,05** | sono médio **8,59 h**
- **Maior consumo médio de café:** ocorre na combinação **`High` × `Poor`** (`2,98`), seguida por **`Medium` × `Fair`** (`2,76`).
- **Interpretação com base no tamanho dos grupos:** o grupo com maior consumo médio (`High × Poor`) é o **menor** dos quatro grupos existentes (menos de `10%` da amostra), enquanto mais da metade das pessoas (`56,37%`) concentra-se em `Low × Good`. As outras 8 combinações têm contagem zero porque, na geração sintética da base, `Stress_Level` e `Sleep_Quality` seguem as mesmas faixas de `Sleep_Hours` (`High` ↔ `Poor`, `Medium` ↔ `Fair`, `Low` ↔ `Good`/`Excellent`).

### Q10 — Possíveis associações com problemas de saúde

Considerando **somente os registros em que `Health_Issues` está preenchido** (`Mild`, `Moderate` e `Severe`), comparar quantidade de registros, médias de `Coffee_Intake`, `Caffeine_mg`, `BMI` e `Heart_Rate`, e a distribuição percentual de `Smoking`.

#### Filtro dos registros preenchidos e resumo por gravidade

In [ ]:
df_saude = df[df["Health_Issues"] != ROTULO_NAO_INFORMADO].copy()

print(f"Registros com Health_Issues preenchido: {len(df_saude)} de {len(df)} ({len(df_saude) / len(df) * 100:.{CASAS_DECIMAIS}f}%)")

resumo_saude = (
    df_saude.groupby("Health_Issues", observed=True)
    .agg(
        Registros=("Coffee_Intake", "count"),
        **{
            "Café (média)": ("Coffee_Intake", "mean"),
            "Café (mediana)": ("Coffee_Intake", "median"),
            "Cafeína mg (média)": ("Caffeine_mg", "mean"),
            "BMI (média)": ("BMI", "mean"),
            "Heart_Rate (média)": ("Heart_Rate", "mean"),
            "Fumantes (%)": ("Smoking", lambda s: s.mean() * 100),
        },
    )
    .assign(**{"% preenchidos": lambda t: t["Registros"] / len(df_saude) * 100})
    .round(CASAS_DECIMAIS)
)
resumo_saude

#### Distribuição percentual de tabagismo (`Smoking`)

In [ ]:
dist_smoking = (
    pd.crosstab(df_saude["Health_Issues"], df_saude["Smoking"], normalize="index") * 100
).rename(columns={0: "Não fumante (0) %", 1: "Fumante (1) %"}).round(CASAS_DECIMAIS)

dist_smoking

#### Visualização

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(FIG_LARGURA, FIG_ALTURA))
categorias_saude = resumo_saude.index.astype(str)
cores_saude = sns.color_palette(n_colors=len(categorias_saude))

# Painel 1: média de Coffee_Intake
b1 = ax1.bar(categorias_saude, resumo_saude["Café (média)"], color=cores_saude)
ax1.bar_label(b1, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.set_title("Média de Coffee_Intake")
ax1.set_ylabel("Média")
ax1.set_ylim(0, resumo_saude["Café (média)"].max() * (1 + MARGEM_TOPO_GRAFICO))

# Painel 2: média de BMI
b2 = ax2.bar(categorias_saude, resumo_saude["BMI (média)"], color=cores_saude)
ax2.bar_label(b2, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax2.set_title("Média de BMI")
ax2.set_ylabel("IMC médio")
ax2.set_ylim(0, resumo_saude["BMI (média)"].max() * (1 + MARGEM_TOPO_GRAFICO))

# Painel 3: distribuição percentual de Smoking
nao_fum = dist_smoking["Não fumante (0) %"]
fum = dist_smoking["Fumante (1) %"]
ax3.bar(categorias_saude, nao_fum, label="Não fumante (0)", color="#9ecae1")
b3_fum = ax3.bar(categorias_saude, fum, bottom=nao_fum, label="Fumante (1)", color="#de2d26")
ax3.bar_label(
    b3_fum,
    labels=[f"{v:.{CASAS_DECIMAIS}f}%" for v in fum],
    label_type="center",
    color="white",
    fontsize="small",
)
ax3.set_title("Distribuição de Smoking (%)")
ax3.set_ylabel("Percentual (%)")
ax3.set_ylim(0, 100 * (1 + MARGEM_TOPO_GRAFICO))
ax3.legend(loc="upper right", fontsize="small")

for ax in (ax1, ax2, ax3):
    ax.set_xlabel("Health_Issues")
    ax.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Quantidade de registros (entre os `4.059` preenchidos):** `Mild` reúne **3.579** (`88,17%`), `Moderate` reúne **463** (`11,41%`) e `Severe` tem apenas **17** registros (`0,42%`).
- **Diferenças observadas:**
  - **`BMI` e `Heart_Rate`:** sobem de forma contínua conforme a gravidade — o `BMI` médio vai de **24,36** (`Mild`) para **27,26** (`Moderate`) e **31,98** (`Severe`), enquanto `Heart_Rate` passa de **70,80** para **71,34** e **72,94** bpm.
  - **`Smoking`:** a proporção de fumantes (`1`) cresce moderadamente de **19,22%** (`Mild`, com `80,78%` não fumantes) para **22,89%** (`Moderate`, `77,11%` não fumantes) e **23,53%** (`Severe`, `76,47%` não fumantes).
  - **`Coffee_Intake` e `Caffeine_mg`:** não seguem tendência crescente contínua — sobem levemente de `Mild` (**2,69** / **255,65 mg**) para `Moderate` (**2,80** / **266,47 mg**), mas **caem** em `Severe` (**2,49** / **236,06 mg**, abaixo até de `Mild`).
- **O café causa problemas de saúde?** **Não é possível afirmar causalidade**, por quatro razões:
  1. **Associação não é causalidade:** a análise exploratória apenas descreve padrões observados, sem desenho experimental nem controle de fatores externos.
  2. **Ausência de padrão crescente para o café:** o grupo `Severe` consome menos café em média do que `Mild` e `Moderate`.
  3. **Amostra muito pequena em `Severe` e alta proporção de ausentes:** `Severe` tem apenas `17` pessoas (qualquer mudança de 1 fumante altera o percentual em quase 6 pontos), e `59,41%` do dataset original está como `Não informado` em `Health_Issues`. Fatores como `BMI` e idade variam bem mais entre os grupos.
  4. **Dados sintéticos:** o conjunto foi gerado artificialmente para estudo e não representa evidência clínica.

## 7. LGPD e ética

Quais dados do dataset são pessoais ou sensíveis, quais riscos éticos o tratamento dessas informações traz e como o projeto se adequa aos princípios da LGPD (Lei nº 13.709/2018).

> O dataset é **sintético**: nenhum registro corresponde a uma pessoa real. Mesmo assim, a análise trata os dados como se fossem reais, porque a mesma estrutura de colunas, coletada de pessoas de verdade, estaria sujeita à LGPD.

### Classificação dos dados segundo a LGPD

- **Dado pessoal** (art. 5º, I): informação relacionada a pessoa natural identificada ou identificável.
- **Dado pessoal sensível** (art. 5º, II): entre outros, dado referente à saúde. Exige uma base legal mais restrita (art. 11).
- **Identificador direto:** aponta para um registro específico (`ID`).
- **Quase-identificador:** sozinho não identifica ninguém, mas combinado com outros pode identificar (`Age`, `Gender`, `Country`, `Occupation`).

In [ ]:
QUASE_IDENTIFICADORES = ["Age", "Gender", "Country", "Occupation"]

categorias_lgpd = {
    "ID": "Identificador direto",
    "Age": "Dado pessoal (quase-identificador)",
    "Gender": "Dado pessoal (quase-identificador)",
    "Country": "Dado pessoal (quase-identificador)",
    "Occupation": "Dado pessoal (quase-identificador)",
    "Coffee_Intake": "Dado pessoal (hábito)",
    "Caffeine_mg": "Dado pessoal (hábito)",
    "Physical_Activity_Hours": "Dado pessoal (hábito)",
    "Sleep_Hours": "Dado sensível (saúde)",
    "Sleep_Quality": "Dado sensível (saúde)",
    "Stress_Level": "Dado sensível (saúde)",
    "BMI": "Dado sensível (saúde)",
    "Heart_Rate": "Dado sensível (saúde)",
    "Health_Issues": "Dado sensível (saúde)",
    "Smoking": "Dado sensível (saúde)",
    "Alcohol_Consumption": "Dado sensível (saúde)",
}

classificacao_lgpd = pd.Series(categorias_lgpd).to_frame("Categoria LGPD")
classificacao_lgpd

In [ ]:
classificacao_lgpd["Categoria LGPD"].value_counts().to_frame("Quantidade de colunas")

### Risco de reidentificação

Remover o `ID` (feito no ETL) não basta para anonimizar. Quem conhece a idade, o gênero, o país e a ocupação de alguém pode encontrar o registro dessa pessoa se a combinação for rara, e com isso descobrir os dados de saúde dela.

A medida usada aqui é o **k-anonimato**: cada combinação de quase-identificadores deve aparecer em pelo menos `K_ANONIMATO` registros. Quanto menor o grupo, mais fácil reidentificar; um grupo de tamanho 1 identifica a pessoa.

In [ ]:
def medir_risco(dados, colunas, k=K_ANONIMATO):
    """Agrupa os registros pelas colunas informadas e mede quantos ficam em grupos menores que k."""
    tamanhos = dados.groupby(colunas, observed=True).size()
    em_risco = tamanhos[tamanhos < k].sum()
    # dtype object mantém as contagens como inteiros ao lado do percentual
    return pd.Series(dtype=object, data={
        "Registros": len(dados),
        "Combinações distintas": len(tamanhos),
        "Registros únicos na combinação": (tamanhos == 1).sum(),
        f"Registros em grupos com menos de {k}": em_risco,
        "% em risco": round(em_risco / len(dados) * 100, 2),
    })


medir_risco(df, QUASE_IDENTIFICADORES).to_frame("Idade exata")

### Medidas de proteção: generalização e supressão

Duas técnicas de anonimização aplicadas a uma versão do DataFrame destinada a compartilhamento:

1. **Generalização:** a idade exata é substituída por faixas etárias (limites definidos em `FAIXAS_ETARIAS`, no `.env`).
2. **Supressão:** os registros que, mesmo com as faixas, continuam em grupos com menos de `K_ANONIMATO` pessoas são retirados.

As análises do notebook continuam usando `df`, porque questões como a Q3 e a Q8 precisam da idade exata. `df_compartilhavel` é a versão que poderia sair do ambiente de análise.

> **Por que a última faixa vai de 45 a 80 anos:** há poucos registros acima de 60 anos (185). Com uma faixa separada de 60 a 80, todos eles caíam em grupos pequenos e a supressão excluía a faixa inteira, deixando os idosos fora dos dados compartilhados. Juntar as faixas reduz a precisão da idade, mas evita apagar um grupo da população.

In [ ]:
rotulos_faixas = [f"{inicio}–{fim - 1}" for inicio, fim in zip(FAIXAS_ETARIAS[:-1], FAIXAS_ETARIAS[1:])]
quase_identificadores_generalizados = ["Faixa_Etaria", "Gender", "Country", "Occupation"]

# Generalização: idade exata → faixa etária
df_generalizado = df.copy()
df_generalizado["Faixa_Etaria"] = pd.cut(
    df_generalizado["Age"], bins=FAIXAS_ETARIAS, labels=rotulos_faixas, right=False
)
df_generalizado = df_generalizado.drop(columns="Age")

# Supressão: retira os registros que continuam em grupos menores que K_ANONIMATO
tamanho_do_grupo = df_generalizado.groupby(
    quase_identificadores_generalizados, observed=True
)["Gender"].transform("size")
df_compartilhavel = df_generalizado[tamanho_do_grupo >= K_ANONIMATO].reset_index(drop=True)

pd.DataFrame({
    "Idade exata": medir_risco(df, QUASE_IDENTIFICADORES),
    "Faixa etária": medir_risco(df_generalizado, quase_identificadores_generalizados),
    "Faixa etária + supressão": medir_risco(df_compartilhavel, quase_identificadores_generalizados),
})

In [ ]:
df_compartilhavel["Faixa_Etaria"].value_counts(sort=False).to_frame("Registros")

### Riscos éticos

| Risco | Como aconteceria |
|---|---|
| **Reidentificação** | Cruzar idade, gênero, país e ocupação com outra fonte (rede social, cadastro de empresa) e chegar à pessoa e aos seus dados de saúde. |
| **Discriminação** | Seguradoras, planos de saúde ou empregadores usarem `BMI`, `Health_Issues`, `Smoking` ou `Alcohol_Consumption` para negar cobertura, cobrar mais ou recusar candidatos. |
| **Desvio de finalidade** | Dados coletados para um estudo sobre café e sono serem usados para publicidade ou perfilamento de consumo. |
| **Vazamento** | Acesso indevido ao arquivo expor dados sensíveis de milhares de pessoas. |
| **Conclusões causais indevidas** | Apresentar associações como causa ("café causa problemas de saúde") e estigmatizar grupos, como fumantes ou certas ocupações. |
| **Viés em modelos** | Um modelo como o da Q8 ser usado para decidir algo sobre uma pessoa, reproduzindo distorções da amostra (por exemplo, o grupo `Other` em `Gender` tem poucos registros). |

### Adequação aos princípios da LGPD

**Base legal.** Por envolver dados de saúde, um tratamento real exigiria uma das hipóteses do art. 11, como o consentimento específico e destacado do titular ou a realização de estudos por órgão de pesquisa, garantida, sempre que possível, a anonimização dos dados sensíveis. Dados anonimizados de forma irreversível deixam de ser considerados dados pessoais (art. 12).

| Princípio (art. 6º) | Como o projeto aplica |
|---|---|
| Finalidade | Os dados são usados só para a análise exploratória e acadêmica descrita no enunciado. |
| Adequação | As análises têm relação direta com o tema do estudo (café, sono, estresse e saúde). |
| Necessidade | Minimização: o `ID` é removido no ETL e a idade é generalizada na versão compartilhável. |
| Livre acesso e transparência | O README e o notebook documentam quais dados são usados, como e para quê. |
| Qualidade dos dados | O ETL verifica ausentes, duplicados e categorias sem alterar o arquivo original. |
| Segurança e prevenção | Repositório privado, acesso restrito aos integrantes e compartilhamento apenas de `df_compartilhavel` (art. 46). |
| Não discriminação | Os resultados são apresentados como associações em dados sintéticos, nunca como critério para decisões sobre pessoas. |
| Responsabilização | As decisões de tratamento ficam registradas no código, no histórico do Git e nos pull requests. |

### Resposta

- **Dados sensíveis:** 8 das 16 colunas são dados referentes à saúde (`Sleep_Hours`, `Sleep_Quality`, `Stress_Level`, `BMI`, `Heart_Rate`, `Health_Issues`, `Smoking` e `Alcohol_Consumption`). `ID` é identificador direto, e `Age`, `Gender`, `Country` e `Occupation` são quase-identificadores.
- **Remover o ID não anonimiza:** com a idade exata, 3.084 registros são únicos na combinação de quase-identificadores e 8.910 (89,1%) ficam em grupos com menos de 5 pessoas.
- **Generalização e supressão funcionam:** com as faixas etárias (18–29, 30–44 e 45–80), os registros em risco caem para 237 (2,37%); retirando esses registros, todos os grupos passam a ter pelo menos 5 pessoas e 9.763 registros (97,6%) são preservados, sem excluir nenhuma faixa etária.
- **Principais riscos éticos:** reidentificação, discriminação por seguradoras ou empregadores, desvio de finalidade, vazamento e conclusões causais indevidas.
- **Adequação à LGPD:** base legal do art. 11 para dados de saúde, anonimização sempre que possível (art. 12), minimização, segurança no armazenamento e transparência sobre o tratamento. Como o dataset é sintético, essas medidas demonstram como o projeto se comportaria com dados reais.

## 8. Hipóteses

Cada hipótese: enunciado → análise → resultado → conclusão.

### Hipótese 1

In [ ]:
# TODO

### Hipótese 2

In [ ]:
# TODO

### Hipótese 3

In [ ]:
# TODO

### Hipótese 4

In [ ]:
# TODO

## 9. Conclusões